This notebook prepares the dataset used for training the object detection model. Images were selected from the PASCAL VOC 2012 `train_val` split and manually annotated in CVAT for four target classes: car, bicycle, bus, and motorbike. The exported Yolo format annotations are organized into training, validation, and test subsets. The notebook also verifies that the dataset structure is correct and that every image has a corresponding label file before training.

In [1]:
import os
import random
import shutil
from pathlib import Path

In [2]:
SEED = 42
random.seed(SEED)
print("Seed set to", SEED)

Seed set to 42


In [3]:
dataset = Path("../data").resolve()

print("Dataset path:", dataset)
print("Dataset exists:", dataset.exists())
print("Images folder exists:", (dataset / "images").exists())
print("Labels folder exists:", (dataset / "labels").exists())

images = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    images.extend((dataset / "images").glob(ext))

print("Total raw images found:", len(images))

label_files = list((dataset / "labels").glob("*.txt"))
print("Total raw labels found:", len(label_files))

Dataset path: C:\Users\ismay\OneDrive\Desktop\Asan3\VOC_Detection_IsmayilMirzaaghayev\data
Dataset exists: True
Images folder exists: True
Labels folder exists: True
Total raw images found: 126
Total raw labels found: 126


In [ ]:
dataset = Path("../data").resolve()
images_dir = dataset / "images"
labels_dir = dataset / "labels"

images = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    images.extend(images_dir.glob(ext))

print("Total images found:", len(images))

random.shuffle(images)

n = len(images)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train = images[:train_end]
val = images[train_end:val_end]
test = images[val_end:]

for split in ["train", "val", "test"]:
    (images_dir / split).mkdir(parents=True, exist_ok=True)
    (labels_dir / split).mkdir(parents=True, exist_ok=True)

def move_files(files, split):

    """
    moves image files and their matching Yolo label files into a target split folder.

    Args:
        files (list[Path]): List of image file paths.
        split (str): Dataset split name ('train', 'val', or 'test').

    Returns:
        None
    """
    
    for img in files:
        label = labels_dir / f"{img.stem}.txt"

        if not label.exists():
            print("Missing label:", img.name)
            continue

        shutil.move(str(img), str(images_dir / split / img.name))
        shutil.move(str(label), str(labels_dir / split / label.name))

move_files(train, "train")
move_files(val, "val")
move_files(test, "test")

print("Dataset split completed.")

Total images found: 126
Dataset split completed.


In [5]:
%%writefile ../data/data.yaml
path: ../data

train: images/train
val: images/val
test: images/test

names:
  0: car
  1: bicycle
  2: bus
  3: motorbike

Writing ../data/data.yaml


In [6]:
dataset = Path("../data").resolve()

print("Dataset exists:", dataset.exists())
print("Images folder exists:", (dataset / "images").exists())
print("Labels folder exists:", (dataset / "labels").exists())
print("data.yaml exists:", (dataset / "data.yaml").exists())

for split in ["train", "val", "test"]:
    img_dir = dataset / "images" / split
    lbl_dir = dataset / "labels" / split

    img_count = (
        len(list(img_dir.glob("*.jpg"))) +
        len(list(img_dir.glob("*.jpeg"))) +
        len(list(img_dir.glob("*.png")))
    )
    lbl_count = len(list(lbl_dir.glob("*.txt")))

    print(f"{split}: images={img_count}, labels={lbl_count}")

Dataset exists: True
Images folder exists: True
Labels folder exists: True
data.yaml exists: True
train: images=100, labels=100
val: images=13, labels=13
test: images=13, labels=13
